# II.18：哪一个中项最近？

本Notebook是现代教学重构，不认证现实原因。相邻步骤由输入声明；集合真值与解释资格分开。实际保存的输出来自标准库exec，并非Jupyter内核。

In [1]:
from model import ChainCertificate, validate, finite_middle_candidates, nearest_choices, transitive_closure, enumerate_four_sets
extensions = {'D': {0}, 'C': {0, 1}, 'B': {0, 1, 2}, 'A': {0, 1, 2, 3}}
steps = {('D', 'C'), ('C', 'B'), ('B', 'A')}
chain = ChainCertificate(('D', 'C', 'B', 'A'))
print(validate(chain, extensions, steps))

{'subject': 'D', 'target': 'A', 'finite_inclusion_holds': True, 'subject_nonempty': True, 'internal_middles': ['C', 'B'], 'nearest_declared_reason': 'C', 'direct_endpoint_convention': False, 'certifies_real_explanation': False}


## 移动主体，不改目标A

最近者随被解释主体变化。B端点不生成B→B，也不证明直接前提。

In [2]:
for subject in ('D', 'C', 'B'):
    result = validate(chain.suffix(subject), extensions, steps)
    print(subject, result['nearest_declared_reason'], result['internal_middles'])

D C ['C', 'B']
C B ['B']
B B []


## 外延不决定顺序

C与B都满足D⊆M⊆A。若四项同外延，另一个链序同样通过集合检查，却选出不同名字。

In [3]:
print('候选中项', finite_middle_candidates('D', 'A', extensions, ('C', 'B')))
equal = {name: {0} for name in 'ABCD'}
reverse_middle_order = ChainCertificate(('D', 'B', 'C', 'A'))
extra = steps | {('D', 'B'), ('B', 'C'), ('C', 'A')}
print('两个指定次序', validate(chain, equal, extra)['nearest_declared_reason'], validate(reverse_middle_order, equal, extra)['nearest_declared_reason'])

候选中项 ('C', 'B')
两个指定次序 C B


## 多路与shortcut

同一问题的多个给定证书可能没有唯一最近名字。闭包中的D→A只是推导可达，不是新发现的直接原因。

In [4]:
other = ChainCertificate(('D', 'E', 'B', 'A'))
validate(other, dict(extensions, E={0, 1}), steps | {('D','E'), ('E','B')})
print('最近名字集合', sorted(nearest_choices([chain, other], 'D', 'A')))
print('D→A已声明?', ('D', 'A') in steps)
print('D→A在闭包?', ('D', 'A') in transitive_closure(steps))

最近名字集合 ['C', 'E']
D→A已声明? False
D→A在闭包? True


## 有限枚举与独立计数

每个元素四集合成员状态有16种，其中满足包含链的只有5种。3元素论域预期4096个总赋值、125个链成立、61个D非空。有限核验不替代一般证明。

In [5]:
result = enumerate_four_sets(3)
expected = {'universe_size': 3, 'assignments': 16**3, 'chain_satisfying': 5**3, 'nonempty_subject': 5**3 - 4**3, 'transitivity_counterexamples': 0}
if result != expected:
    raise RuntimeError('independent count mismatch')
print(result)

{'universe_size': 3, 'assignments': 4096, 'chain_satisfying': 125, 'nonempty_subject': 61, 'transitivity_counterexamples': 0}


## 复盘

一个链证书能检查有限集合真值，却无法判断原因适当性或相邻步骤的不可再分性。最近理由是相对于输入解释链和当前问题而言；它不是从闭包最短路得到的科学原因。